# Step 00: Dataset Integrity & 3-Way Train/Val/Test Partitioning

Ingests the **TI IWR6843 mmWave Radar Dataset** (102 CSV files: 51 Falls, 51 ADLs across subjects **Areeb**, **Raffay**, **Towsif**) and partitions them into **Train**, **Validation**, and **Test** sets with zero cross-subject data leakage.


In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
root_dir = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root_dir) not in sys.path: sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.dataset_parser import get_all_recording_files, get_loso_splits, extract_subject, extract_label
from src.config import DATASET_CONFIGS

sns.set_theme(style="whitegrid")
print(f"Dataset: {DATASET_CONFIGS['ti'].name}")


In [ ]:
# 1. Ingestion Audit (102 CSV Files)
all_files = get_all_recording_files()
records = []
for f in all_files:
    subj = extract_subject(f)
    lbl = extract_label(f)
    df_raw = pd.read_csv(f)
    records.append({
        "file": f.name,
        "subject": subj,
        "label": lbl,
        "class_name": "Fall" if lbl == 1 else "Non-Fall (ADL)",
        "total_frames": df_raw["frame"].nunique(),
        "total_points": len(df_raw)
    })
df_meta = pd.DataFrame(records)
print(f"Discovered {len(df_meta)} CSV recording files (51 Falls vs 51 ADLs).")
df_meta.head(10)


In [ ]:
# 2. 3-Way Train / Validation / Test Partitioning per Fold
loso_splits = get_loso_splits()
print("=== 3-Fold LOSO Train / Validation / Test Partitioning Scheme ===")
for fold_info in loso_splits:
    f_idx = fold_info["fold"]
    tr_subs = ", ".join(fold_info["train_subjects"])
    te_sub = fold_info["test_subject"]
    tr_f = len(fold_info['train_files'])
    va_f = len(fold_info['val_files'])
    te_f = len(fold_info['test_files'])
    print(f"Fold {f_idx} [Held-Out Test Subject: {te_sub:6s}]: "
          f"Train = {tr_f} files (27 Falls, 27 ADLs) | "
          f"Val = {va_f} files (7 Falls, 7 ADLs) | "
          f"Test = {te_f} files (17 Falls, 17 ADLs)")


In [ ]:
# 3. Visual Breakdown of 3-Way LOSO Partitioning (54 Train / 14 Val / 34 Test)
fig, ax = plt.subplots(figsize=(10, 4.5), dpi=100)
fold_labels = [f"Fold {f['fold']}\n(Test: {f['test_subject']})" for f in loso_splits]
tr_counts = [len(f["train_files"]) for f in loso_splits]
va_counts = [len(f["val_files"]) for f in loso_splits]
te_counts = [len(f["test_files"]) for f in loso_splits]

x_idx = np.arange(len(fold_labels))
ax.bar(x_idx - 0.25, tr_counts, width=0.25, label="Train Set (54 files: 27 Fall/27 ADL)", color="#3498db")
ax.bar(x_idx, va_counts, width=0.25, label="Val Set (14 files: 7 Fall/7 ADL)", color="#f39c12")
ax.bar(x_idx + 0.25, te_counts, width=0.25, label="Held-Out Test Set (34 files: 17 Fall/17 ADL)", color="#e74c3c")
ax.set_xticks(x_idx)
ax.set_xticklabels(fold_labels)
ax.set_title("3-Fold LOSO 3-Way Partitioning (54 Train / 14 Val / 34 Test)", fontweight="bold")
ax.set_ylabel("Number of CSV Recording Files")
ax.legend()
plt.tight_layout()
plt.show()
